# Demo 18 — Plot annotations (`show_emm_lines`, `show_group_size`)

Two options change nothing about the model and everything about how quickly the
figure can be read. Both are off by default, so a plot only carries the ink you
ask for.

`options.show_emm_lines` extends each group's estimated marginal mean across the
whole panel, in that group's own colour. The EMM is already marked by the white
dot, but reading one group's level against the *other* groups meant comparing dot
heights by eye across the panel; the line turns that into a direct read-off. Each
panel uses its own EMMs, so the lines move between panels.

The option doubles as the line style: `True` gives the default dotted line, which
recedes behind the violins so a line crossing one cannot be mistaken for plotted
data, while `'-'`, `'--'`, `':'` and `'-.'` (or the names `'solid'`, `'dashed'`,
`'dotted'`, `'dashdot'`) pick one explicitly.

`options.show_group_size` labels each group with the number of observations behind
it, just above the violin (or above the CI bar in bar style) — useful wherever the
cells are unbalanced, or where outlier removal has thinned some groups more than
others. The significance brackets stack above the labels and keep a visible gap
from them, so switching the labels on pushes the brackets up rather than colliding
with them.

The model is the crossed two-way design of Demo 3, with the roles of the two
factors swapped so the ordered dose is on the x-axis and the supplement makes the
panels:

    len ~ dose * supp

Two runs of the same model follow: bare, then annotated.

## Run on Google Colab

On [Google Colab](https://colab.research.google.com)? Run the cell below first —
it installs kbstatpy, its R packages, and the demo data (~3–5 min the first time).
It is a no-op when you run this notebook locally from the kbstatpy source tree.
Then run the cells below to see the tables and figures rendered inline.

In [ ]:
# Google Colab only: install kbstatpy + its R packages + the demo data.
# (Does nothing when the notebook runs locally from the source tree.)
import sys
if 'google.colab' in sys.modules:
    !curl -sSL https://raw.githubusercontent.com/kimbostroem/kbstatpy/master/demos/colab_setup.sh | bash

## Setup

In [ ]:
import os

from kbstatpy import Kbstat, KbstatOptions

## The model

`len ~ dose * supp`, shared by both runs. `out_dir` is empty, so results render
inline only.

In [ ]:
def base_options():
    o = KbstatOptions()
    o.in_file     = os.path.join(o.demo_dir, 'data/toothgrowth.csv')
    o.out_dir     = ''            # inline only; set a folder to also save
    o.y           = 'len'
    o.y_units     = 'mm'
    o.x           = 'dose, supp'  # dose on the x-axis, supp as panels
    o.interaction = 'dose, supp'
    o.x_order     = 'dose: low, medium, high'
    o.rename      = ('len -> ToothLength; supp -> Supplement; dose -> Dose; '
                     'supp: OJ -> orange_juice, VC -> vitamin_c')
    return o

## 1. The defaults

No reference lines, no counts: the EMM of each group is the white dot, and the
group sizes are not shown.

In [ ]:
bare = Kbstat(base_options())
bare.run();

## 2. Annotated

`show_emm_lines = True` draws each group's EMM across its panel (dotted, in the
group's colour) and `show_group_size = True` labels each violin with its count.
The model, the estimates and the brackets are identical to run 1 — only the
annotation differs, and the brackets have moved up to clear the new labels.

In [ ]:
annotated = base_options()
annotated.show_emm_lines  = True      # or a style: '-', '--', ':', '-.'
annotated.show_group_size = True

kb = Kbstat(annotated)
kb.run();

## 3. Picking a line style

Solid reads calmest and makes the group colours easiest to attribute; dotted (the
default) recedes furthest, so a line crossing a violin cannot be mistaken for
plotted data. Dashed sits between the two.

In [ ]:
solid = base_options()
solid.show_emm_lines  = 'solid'       # same as '-'
solid.show_group_size = True

kb_solid = Kbstat(solid)
kb_solid.run();

## Interpretation

- The dose lines climb in both panels, and the gaps between them are the
  dose effect *within* that supplement.
- Compare the two panels: under ascorbic acid the high-dose line clears the whole
  medium-dose violin, while under orange juice the two overlap by a wide margin.
  That difference in spacing between panels *is* the dose × supp interaction —
  visible in the figure, and confirmed by the interaction term in the ANOVA table.
- The counts (`n=10` everywhere here, a balanced design) matter more in unbalanced
  data or after outlier removal, where a group's n is no longer obvious. Demo 11
  shows them on a bar plot with genuinely unequal cells.
- Neither option touches the model: the ANOVA, the post-hoc table and the EMMs are
  the same in all three runs above.